In [4]:
!pip install aif360
from aif360.datasets import AdultDataset
from aif360.metrics import BinaryLabelDatasetMetric
import pandas as pd

pip install 'aif360[Reductions]'
pip install 'aif360[Reductions]'
pip install 'aif360[inFairness]'
pip install 'aif360[Reductions]'


In [5]:
import os
import urllib.request

base_dir = r"C:\Users\DELL\anaconda3\Lib\site-packages\aif360\data\raw\adult"
os.makedirs(base_dir, exist_ok=True)

urls = {
    "adult.data": "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data",
    "adult.test": "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test",
    "adult.names": "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.names"
}

for filename, url in urls.items():
    filepath = os.path.join(base_dir, filename)
    if not os.path.exists(filepath):
        urllib.request.urlretrieve(url, filepath)
        print(f"{filename} téléchargé")
    else:
        print(f"{filename} déjà présent")

dataset = AdultDataset()

adult.data déjà présent
adult.test déjà présent
adult.names déjà présent


In [ ]:
df= dataset.convert_to_dataframe()[0]
df.describe()

In [ ]:
print("Features:", dataset.feature_names)

In [ ]:
print("protected attribute:", dataset.protected_attribute_names)

In [ ]:
print("Favorable label(icome>50k):", dataset.favorable_label)

In [ ]:
print(df.columns)


In [ ]:
print(df.columns)


In [36]:
# Proportion globale >50K
df, _ = dataset.convert_to_dataframe()
prop_global = df['income-per-year'].value_counts(normalize=True)
print(prop_global)

# Proportion par sexe
prop_by_sex = pd.crosstab(
    df['sex'],
    df['income-per-year'],
    normalize='index'
)
print(prop_by_sex)

income-per-year
0.0    0.752156
1.0    0.247844
Name: proportion, dtype: float64
income-per-year       0.0       1.0
sex                                
0.0              0.886424  0.113576
1.0              0.687523  0.312477


In [37]:
print(df.groupby('sex')['income-per-year'].mean())

sex
0.0    0.113576
1.0    0.312477
Name: income-per-year, dtype: float64


CONVERSION EN ARRAY NUMPY

In [38]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from aif360.algorithms.preprocessing import reweighing

In [39]:
x = dataset.features

In [40]:
y = dataset.labels.ravel()

In [41]:
attrs=dataset.protected_attributes

SEPARATION TRAIN/TEST

In [42]:
x_train, x_test, y_train, y_test, attrs_train, attrs_test = train_test_split(
    x, y, attrs, test_size=0.3, random_state=42
)

#Entrainement du model

In [43]:
model = RandomForestClassifier(random_state=42)

In [44]:
model.fit(x_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


#Predition

In [45]:
y_pred = model.predict(x_test)

# CREATION DU DATASET DU TEST

In [46]:
from aif360.datasets import BinaryLabelDataset

#RECONSTRUCTION DU DATASET DE TEST AVEC VRAIES ET PREDITES LABELS

In [47]:
import pandas as pd
feature_names = dataset.feature_names
df_test = pd.DataFrame(x_test, columns=feature_names)
df_test['income-per-year'] = y_test
df_test['sex'] = attrs_test[:,0]


#METRIQUE D EQUITE DE BASE

In [48]:
test_dataset = BinaryLabelDataset(
    df=df_test,
    label_names=['income-per-year'],
    protected_attribute_names=['sex'],
    favorable_label=1,
    unfavorable_label=0
)

In [49]:
df_pred = df_test.copy()
df_pred['income-per-year'] = y_pred.reshape(-1, 1)

pred_dataset = BinaryLabelDataset(
    df=df_pred,
    label_names=['income-per-year'],
    protected_attribute_names=['sex'],
    favorable_label=1,
    unfavorable_label=0
)
metric = BinaryLabelDatasetMetric(
    pred_dataset,
    unprivileged_groups=[{'sex': 0}],  # Female
    privileged_groups=[{'sex': 1}]     # Male
)

In [50]:
print(x_test.shape, y_test.shape, attrs_test.shape)
print(pred_dataset.labels.shape, pred_dataset.protected_attributes.shape)

(13567, 98) (13567,) (13567, 2)
(13567, 1) (13567, 1)


In [51]:
print("Statistical parity difference:", metric.statistical_parity_difference())

Statistical parity difference: -0.10551991254350071


In [52]:
print("Diparate impact:", metric.disparate_impact())

Diparate impact: 0.5467025881216538
